<a href="https://colab.research.google.com/github/Aliahmadjangohar/Aliahmadjangohar/blob/main/CIFAR_100_IMAGE_PROCESSING_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# COMPLETE NOTEBOOK FOR CIFAR-100 IMAGE PROJECTION MODEL TRAINING
# =============================================================================

import random
import requests
from io import BytesIO
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
import os
import sys

In [ ]:
# =============================================================================
# DATASET & MODEL - COMPLETED
# =============================================================================

class CIFAR100Filtered(Dataset):
    """CIFAR-100 dataset wrapper with preprocessing and train/val split support."""

    def __init__(self, root="./data", split="train", transform=None):
        assert split in ["train", "val"], "Split must be 'train' or 'val'"

        if transform is None:
            transform = transforms.Compose([
                transforms.Resize(224),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])

        self.dataset = datasets.CIFAR100(
            root=root,
            train=(split == "train"),
            download=True,
            transform=transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]


class ImageEncoder(nn.Module):
    """MobileNetV3-based image encoder with trainable projection head."""

    def __init__(self, proj_dim=64, device="cuda"):
        super().__init__()
        self.device = device

        # Load pretrained MobileNetV3-Small
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        # Extract all layers except classifier
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.backbone.to(device)
        self.backbone.eval()

        # Freeze backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Create projection head
        self.projection = nn.Sequential(
            nn.Linear(576, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, proj_dim)
        ).to(device)

    def forward(self, x):
        # Extract features with frozen backbone
        with torch.no_grad():
            features = self.backbone(x)
            features = features.flatten(1)  # (batch_size, 576)

        # Project through trainable head
        projections = self.projection(features)

        return features, projections


In [ ]:
# =============================================================================
# DATA & TRAINING UTILITIES - COMPLETED
# =============================================================================

def filter_dataset_indices(dataset, valid_labels):
    """Return indices of samples with labels in valid_labels set."""
    return [i for i, label in enumerate(dataset.dataset.targets) if label in valid_labels]

def create_data_splits(indices, val_ratio=0.2, seed=42):
    """Split indices into train/val sets."""
    np.random.seed(seed)
    indices = np.array(indices)
    np.random.shuffle(indices)
    split_idx = int((1 - val_ratio) * len(indices))
    return indices[:split_idx].tolist(), indices[split_idx:].tolist()

def create_dataloaders(train_idx, val_idx, test_idx, batch_sizes):
    """Create train, val, and test dataloaders."""
    datasets = {
        'train': Subset(CIFAR100Filtered(split="train"), train_idx),
        'val': Subset(CIFAR100Filtered(split="train"), val_idx),
        'test': Subset(CIFAR100Filtered(split="val"), test_idx)
    }
    return {k: DataLoader(v, batch_size=batch_sizes['train' if k == 'train' else 'eval'],
                         shuffle=(k == 'train'), num_workers=2) for k, v in datasets.items()}

def compute_contrastive_loss(visual_proj, text_emb, temperature):
    """
    Compute symmetric InfoNCE (contrastive) loss for vision-language alignment.
    """
    # Normalize embeddings
    visual_norm = F.normalize(visual_proj, p=2, dim=1)
    text_norm = F.normalize(text_emb, p=2, dim=1)

    # Compute similarity matrix
    logits = torch.matmul(visual_norm, text_norm.T) / temperature
    batch_size = visual_proj.size(0)

    # Create labels
    labels = torch.arange(batch_size, device=visual_proj.device)

    # Compute losses
    i2t_loss = F.cross_entropy(logits, labels)
    t2i_loss = F.cross_entropy(logits.T, labels)

    # Symmetric loss
    loss = (i2t_loss + t2i_loss) / 2
    return loss

def run_epoch(model, dataloader, text_emb, class_words, label_to_word, optimizer, temperature, device, mode='train'):
    """
    Run one epoch of training or evaluation for contrastive vision-language learning.
    """
    if mode == 'train':
        model.train()
        torch.set_grad_enabled(True)
    else:
        model.eval()
        torch.set_grad_enabled(False)

    total_loss = 0
    total_sim = 0
    count = 0

    pbar = tqdm(dataloader, desc=f"{mode.capitalize()} Epoch")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        # Get visual embeddings
        _, visual_proj = model(images)

        # Get corresponding text embeddings
        batch_text_idx = labels.cpu().numpy()
        batch_text_emb = torch.stack([text_emb[idx] for idx in batch_text_idx]).to(device)

        # Compute loss
        loss = compute_contrastive_loss(visual_proj, batch_text_emb, temperature)

        if mode == 'train':
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Compute similarity for monitoring
        visual_norm = F.normalize(visual_proj, p=2, dim=1)
        text_norm = F.normalize(batch_text_emb, p=2, dim=1)
        batch_sim = (visual_norm * text_norm).sum(dim=1).mean().item()

        # Accumulate statistics
        total_loss += loss.item() * len(images)
        total_sim += batch_sim * len(images)
        count += len(images)

        # Update progress bar
        pbar.set_postfix({'loss': loss.item(), 'sim': batch_sim})

    return total_loss / count, total_sim / count

def train_with_early_stopping(model, dataloaders, text_emb, class_words, label_to_word, config, device):
    """
    Train model with early stopping based on validation similarity.
    """
    # Create optimizer only for projection parameters
    optimizer = torch.optim.AdamW(
        model.projection.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )

    # Create learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['epochs']
    )

    # Initialize tracking variables
    best_val_sim = -float('inf')
    patience_counter = 0
    best_epoch = 0
    best_val_loss = float('inf')
    history = defaultdict(list)

    print(f"\n{'='*70}\nTraining (max {config['epochs']} epochs, patience={config['patience']})\n{'='*70}")

    try:
        for epoch in range(1, config['epochs'] + 1):
            print(f"\nEpoch {epoch}/{config['epochs']}")

            # Training phase
            train_loss, train_sim = run_epoch(
                model, dataloaders['train'], text_emb, class_words, label_to_word,
                optimizer, config['temperature'], device, mode='train'
            )

            # Validation phase
            val_loss, val_sim = run_epoch(
                model, dataloaders['val'], text_emb, class_words, label_to_word,
                None, config['temperature'], device, mode='eval'
            )

            # Step scheduler
            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]

            # Store history
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_similarity'].append(val_sim)
            history['learning_rate'].append(current_lr)

            # Print epoch summary
            print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Sim: {val_sim:.4f}")
            print(f"  Learning Rate: {current_lr:.6f}")

            # Check for improvement
            if val_sim > best_val_sim:
                best_val_sim = val_sim
                best_val_loss = val_loss
                best_epoch = epoch
                patience_counter = 0

                # Save checkpoint
                checkpoint = {
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_loss': val_loss,
                    'val_similarity': val_sim,
                    'class_words': class_words,
                    'text_embeddings': text_emb.cpu(),
                    'history': dict(history),
                    'projection_head': model.projection.state_dict(),
                    'config': config
                }
                torch.save(checkpoint, config['save_path'])
                print(f"  ✓ New best model saved to {config['save_path']}")
            else:
                patience_counter += 1
                print(f"  No improvement for {patience_counter}/{config['patience']} epochs")

            # Early stopping check
            if patience_counter >= config['patience']:
                print(f"\nEarly stopping triggered after {epoch} epochs")
                break

    except Exception as e:
        print(f"\n⚠️ Training interrupted by error: {str(e)}")
        print("Continuing with best saved model...")

    # Load best model
    if os.path.exists(config['save_path']):
        print(f"\nLoading best model from epoch {best_epoch}...")
        checkpoint = torch.load(config['save_path'], map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])

    return dict(history), best_epoch, best_val_sim, best_val_loss


In [ ]:
# =============================================================================
# ANALYSIS FUNCTIONS - COMPLETED
# =============================================================================

def collect_embeddings(model, dataloader, device):
    """Collect all embeddings and labels from dataset."""
    model.eval()
    all_visual, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Collecting embeddings"):
            images = images.to(device)
            _, visual_proj = model(images)
            all_visual.append(F.normalize(visual_proj, p=2, dim=1).cpu())
            all_labels.extend(labels.tolist())
    return torch.cat(all_visual, dim=0).numpy(), all_labels

def compute_alignment_metrics(visual_emb, labels, text_emb, class_words, label_to_word):
    """Compute comprehensive alignment metrics in one pass."""
    # Per-class statistics
    class_sims = defaultdict(list)
    for i, label in enumerate(labels):
        if (word := label_to_word[label]) in class_words:
            sim = np.dot(visual_emb[i], text_emb[class_words.index(word)])
            class_sims[word].append(sim)

    stats = sorted([{
        'word': word, 'mean': np.mean(sims), 'std': np.std(sims),
        'min': np.min(sims), 'max': np.max(sims), 'count': len(sims)
    } for word, sims in class_sims.items()], key=lambda x: x['mean'], reverse=True)

    # Retrieval metrics
    sim_matrix = cosine_similarity(visual_emb, text_emb)
    i2t_recalls = {k: 0 for k in [1, 5, 10]}
    t2i_recalls = {k: 0 for k in [1, 5, 10]}

    # Image-to-text retrieval
    for i, label in enumerate(labels):
        if (word := label_to_word[label]) in class_words:
            correct_idx = class_words.index(word)
            ranking = np.argsort(-sim_matrix[i])
            for k in i2t_recalls:
                if correct_idx in ranking[:k]:
                    i2t_recalls[k] += 1

    # Text-to-image retrieval
    for class_idx, word in enumerate(class_words):
        class_img_idx = [i for i, l in enumerate(labels) if label_to_word[l] == word]
        if class_img_idx:
            ranking = np.argsort(-sim_matrix[:, class_idx])
            for k in t2i_recalls:
                if any(idx in ranking[:k] for idx in class_img_idx):
                    t2i_recalls[k] += 1

    return stats, i2t_recalls, t2i_recalls, sim_matrix

def print_analysis_results(stats, i2t_recalls, t2i_recalls, n_samples, n_classes):
    """Print comprehensive analysis results."""
    print("\n📊 Per-Class Similarity Analysis:")
    print("-" * 70)
    for title, data in [("Top 10 Best Aligned Classes:", stats[:10]),
                        ("Bottom 10 Worst Aligned Classes:", stats[-10:])]:
        print(f"\n{title}")
        for i, s in enumerate(data, 1):
            print(f"{i:2d}. {s['word']:15s} | Mean: {s['mean']:.4f} ± {s['std']:.4f}")

    print("\n📊 Retrieval Performance:")
    print("-" * 70)
    for name, recalls, total in [("Image-to-Text", i2t_recalls, n_samples),
                                 ("Text-to-Image", t2i_recalls, n_classes)]:
        print(f"\n{name} Retrieval (Recall@K):")
        for k, count in recalls.items():
            print(f"  Recall@{k:2d}: {count/total*100:.2f}% ({count}/{total})")

def print_example_retrievals(sim_matrix, labels, class_words, label_to_word, n_examples=5):
    """Print text-based retrieval examples."""
    print("\n📸 Example Image-to-Text Retrievals:")
    print("-" * 70)

    display_idx = np.random.choice(len(labels), size=n_examples, replace=False)

    for idx in display_idx:
        label = labels[idx]
        true_word = label_to_word[label]

        sims = sim_matrix[idx]
        top_5_idx = np.argsort(-sims)[:5]
        top_5_words = [class_words[i] for i in top_5_idx]
        top_5_sims = [sims[i] for i in top_5_idx]

        correct_sim = sims[class_words.index(true_word)]
        correct_rank = np.where(np.argsort(-sims) == class_words.index(true_word))[0][0] + 1

        print(f"\nTest Image #{idx}:")
        print(f"  True class: '{true_word}' (similarity: {correct_sim:.4f}, rank: {correct_rank})")
        print(f"  Top 5 predictions:")
        for rank, (word, sim) in enumerate(zip(top_5_words, top_5_sims), 1):
            marker = "✓" if word == true_word else " "
            print(f"    {rank}. {marker} {word:15s} (similarity: {sim:.4f})")

In [ ]:
# =============================================================================
# VISUALIZATION FUNCTIONS - COMPLETED
# =============================================================================

def create_visualizations(sim_matrix, labels, class_words, label_to_word, test_indices, images=None, names=None, predictions=None):
    """Create all visualizations in one coordinated function."""
    # OOD analysis if provided
    if images and names and predictions:
        print(f"\n📸 Creating OOD visualization for {len(images)} images...")
        n_imgs = len(images)
        n_cols = min(4, n_imgs)
        n_rows = (n_imgs + n_cols - 1) // n_cols

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 7*n_rows))
        axes = [axes] if n_rows == 1 and n_cols == 1 else axes.flatten()

        for i, (img, name, pred) in enumerate(zip(images, names, predictions)):
            axes[i].imshow(img)
            axes[i].axis('off')
            pred_text = f"{name.upper()}\n\nTop matches:\n"
            for rank, (word, sim) in enumerate(zip(pred['words'][:5], pred['sims'][:5]), 1):
                pred_text += f"{rank}. {word} ({sim:.3f})\n"
            axes[i].set_title(pred_text, fontsize=11, ha='center', color='darkblue', fontweight='bold', pad=12)

        for j in range(i+1, len(axes)):
            axes[j].axis('off')
            axes[j].set_visible(False)

        plt.tight_layout()
        plt.savefig('ood_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()

    else:
        print("\n📊 Creating confusion matrix...")
        n_classes = len(class_words)
        conf_matrix = np.zeros((n_classes, n_classes))

        for i, label in enumerate(labels):
            if (word := label_to_word[label]) in class_words:
                true_idx = class_words.index(word)
                pred_idx = np.argmax(sim_matrix[i])
                conf_matrix[true_idx, pred_idx] += 1

        conf_matrix = conf_matrix / (conf_matrix.sum(axis=1, keepdims=True) + 1e-10)

        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(conf_matrix, xticklabels=class_words, yticklabels=class_words,
                    cmap='Blues', ax=ax, cbar_kws={'label': 'Probability'}, square=True)
        ax.set_xlabel('Predicted Class')
        ax.set_ylabel('True Class')
        ax.set_title('Confusion Matrix (All Classes)', fontsize=14, fontweight='bold')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
        plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)
        plt.tight_layout()
        plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.show()

        # Retrieval examples
        print("\n📸 Creating retrieval examples...")
        test_raw = CIFAR100Filtered(split="val", transform=transforms.Compose([
            transforms.Resize(224),
            transforms.ToTensor()
        ]))

        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        axes = axes.flatten()

        for plot_idx in range(min(12, len(labels))):
            ex_idx = random.randint(0, len(labels)-1)
            original_idx = test_indices[ex_idx]
            img, label = test_raw[original_idx]

            axes[plot_idx].imshow(img.permute(1, 2, 0).numpy())
            axes[plot_idx].axis('off')

            true_word = label_to_word[label]
            sims = sim_matrix[ex_idx]
            top_5_idx = np.argsort(-sims)[:5]
            top_5_words = [class_words[i] for i in top_5_idx]
            top_5_sims = [sims[i] for i in top_5_idx]

            # Build prediction text
            pred_text = f"GT: {true_word}\n"
            for rank, (word, sim) in enumerate(zip(top_5_words, top_5_sims), 1):
                marker = "✓" if word == true_word else "✗"
                pred_text += f"{rank}. {marker} {word}: {sim:.2f}\n"

            # Color logic
            if top_5_words[0] == true_word:
                title_color = "green"          # correct top-1
            elif true_word in top_5_words:
                title_color = "#CC8A00"        # amber
            else:
                title_color = "red"            # incorrect

            axes[plot_idx].set_title(
                pred_text, fontsize=9, ha='left',
                fontfamily='monospace',
                color=title_color, fontweight='bold'
            )

        # Hide any unused subplots
        for plot_idx in range(min(12, len(labels)), 12):
            axes[plot_idx].axis('off')
            axes[plot_idx].set_visible(False)

        plt.tight_layout()
        plt.savefig('retrieval_examples.png', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
# =============================================================================
# OOD PROCESSING - COMPLETED
# =============================================================================

def process_ood_images(model, image_urls, text_emb, class_words, device):
    """Download and process OOD images in one function."""
    print(f"\nDownloading {len(image_urls)} OOD test images...")
    images, names = [], []
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'image/*'}

    for desc, url in image_urls.items():
        try:
            response = requests.get(url, timeout=30, headers=headers)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert('RGB')
                images.append(img.resize((224, 224), Image.BILINEAR))
                names.append(desc)
                print(f"  ✓ Downloaded: {desc}")
        except Exception as e:
            print(f"  ✗ Error downloading {desc}: {str(e)[:50]}")

    if not images:
        return [], [], []

    print(f"\n🔬 Processing {len(images)} OOD images...")
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    to_tensor = transforms.ToTensor()

    model.eval()
    with torch.no_grad():
        ood_emb = []
        for img in images:
            img_tensor = normalize(to_tensor(img).unsqueeze(0).to(device))
            _, visual_proj = model(img_tensor)
            ood_emb.append(F.normalize(visual_proj, p=2, dim=1).cpu().numpy()[0])

    ood_emb = np.array(ood_emb)
    predictions = []
    for emb in ood_emb:
        sims = cosine_similarity(emb.reshape(1, -1), text_emb)[0]
        top_5_idx = np.argsort(-sims)[:5]
        predictions.append({
            'words': [class_words[j] for j in top_5_idx],
            'sims': [sims[j] for j in top_5_idx]
        })

    return images, names, predictions

In [ ]:
# =============================================================================
# FINAL REPORTING - COMPLETED
# =============================================================================

def print_final_report(config, test_loss, test_sim, i2t_recalls, t2i_recalls, class_stats,
                      n_train, n_val, n_test, n_classes, batch_size, has_ood, history=None,
                      best_epoch=None, best_val_sim=None, best_val_loss=None, n_vocab_total=None):
    """Print comprehensive final summary report."""
    n_samples = n_test

    print(f"""
📋 Training Configuration:
   ├─ Model: MobileNetV3-Small with projection head
   ├─ Embedding dimension: {config['proj_dim']}
   ├─ Training samples: {n_train:,}, Validation samples: {n_val:,}, Test samples: {n_test:,}
   ├─ Number of training classes: {n_classes}
   {f'├─ Total vocabulary size: {n_vocab_total} words' if n_vocab_total else ''}
   {f'├─ Total epochs trained: {len(history["train_loss"]) if history else 0}, Best epoch: {best_epoch if best_epoch else "N/A"}' if history else ''}
   └─ Early stopping patience: {config['patience']}

🎯 Performance Metrics:
   {f"├─ Best Val Similarity: {best_val_sim:.4f}" if best_val_sim else ""}
   ├─ Test Similarity: {test_sim:.4f}, Test Loss: {test_loss:.4f}
   ├─ Random baseline loss: ~{np.log(batch_size):.2f}
   │
   ├─ Image→Text Recall@1: {i2t_recalls[1]/n_samples*100:.2f}%
   ├─ Image→Text Recall@5: {i2t_recalls[5]/n_samples*100:.2f}%
   ├─ Image→Text Recall@10: {i2t_recalls[10]/n_samples*100:.2f}%
   │
   ├─ Text→Image Recall@1: {t2i_recalls[1]/n_classes*100:.2f}%
   ├─ Text→Image Recall@5: {t2i_recalls[5]/n_classes*100:.2f}%
   └─ Text→Image Recall@10: {t2i_recalls[10]/n_classes*100:.2f}%

📊 Embedding Space Alignment:
   ├─ Mean per-class similarity: {np.mean([s['mean'] for s in class_stats]):.4f} ± {np.std([s['mean'] for s in class_stats]):.4f}
   ├─ Best aligned class: '{class_stats[0]['word']}' ({class_stats[0]['mean']:.4f})
   └─ Worst aligned class: '{class_stats[-1]['word']}' ({class_stats[-1]['mean']:.4f})

💡 Key Insights:
   • The model {'successfully learns' if test_sim > 0.5 else 'attempts to learn'} visual-text alignment
   • {'High' if test_sim > 0.7 else 'Moderate' if test_sim > 0.5 else 'Low'} overall alignment (similarity: {test_sim:.4f})
   • Loss: {test_loss:.2f} vs random baseline ~{np.log(batch_size):.2f}
   • Retrieval performance: {'Good' if i2t_recalls[1]/n_samples > 0.5 else 'Moderate'}
   • Class performance varies (range: {class_stats[-1]['mean']:.4f} to {class_stats[0]['mean']:.4f})
   {f'• OOD predictions use full vocabulary of {n_vocab_total} words' if n_vocab_total else ''}

✅ Model saved to: '{config['save_path']}'
✅ Confusion matrix & retrieval examples saved
{'✅ OOD analysis saved' if has_ood else ''}
""")

In [ ]:
# =============================================================================
# MAIN TRAINING PIPELINE
# =============================================================================

def main():
    """Main training pipeline for CIFAR-100 image projection model."""

    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Check if GPU is available in Colab
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Configuration
    config = {
        'proj_dim': 64,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 50,
        'temperature': 0.07,
        'patience': 10,
        'save_path': 'best_cifar100_projection.pth',
        'batch_sizes': {'train': 64, 'eval': 128}
    }

    # Step 1: Create dummy text embeddings for CIFAR-100 classes
    # In your actual implementation, you would load these from best_skipgram_523words.pth
    print("\n🔧 Creating dummy text embeddings for CIFAR-100 classes...")

    # Get CIFAR-100 class names
    cifar100_classes = [
        'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
        'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
        'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
        'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
        'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
        'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
        'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
        'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
        'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
        'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose',
        'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake',
        'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
        'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout',
        'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman',
        'worm'
    ]

    # Create dummy embeddings (normally you'd load these from your Skip-gram model)
    n_classes = len(cifar100_classes)
    dummy_text_emb = torch.randn(n_classes, config['proj_dim'])
    dummy_text_emb = F.normalize(dummy_text_emb, p=2, dim=1).to(device)

    # Create label to word mapping
    label_to_word = {i: word for i, word in enumerate(cifar100_classes)}
    class_words = cifar100_classes

    # Step 2: Prepare datasets
    print("\n📊 Preparing CIFAR-100 datasets...")

    # Create full dataset indices
    train_dataset = CIFAR100Filtered(split="train")
    test_dataset = CIFAR100Filtered(split="val")

    # Use all classes for now (you can filter specific classes if needed)
    all_train_indices = list(range(len(train_dataset)))
    all_test_indices = list(range(len(test_dataset)))

    # Create train/val split
    train_idx, val_idx = create_data_splits(all_train_indices, val_ratio=0.2, seed=42)

    # Create dataloaders
    dataloaders = create_dataloaders(
        train_idx, val_idx, all_test_indices, config['batch_sizes']
    )

    print(f"  Training samples: {len(train_idx):,}")
    print(f"  Validation samples: {len(val_idx):,}")
    print(f"  Test samples: {len(all_test_indices):,}")
    print(f"  Number of classes: {n_classes}")

    # Step 3: Initialize model
    print("\n🧠 Initializing image encoder model...")
    model = ImageEncoder(proj_dim=config['proj_dim'], device=device)

    # Count parameters
    trainable_params = sum(p.numel() for p in model.projection.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,} (projection head only)")

    # Step 4: Train the model
    print("\n🚀 Starting training...")
    history, best_epoch, best_val_sim, best_val_loss = train_with_early_stopping(
        model, dataloaders, dummy_text_emb, class_words, label_to_word, config, device
    )

    # Step 5: Test the model
    print("\n🧪 Testing model on test set...")
    test_loss, test_sim = run_epoch(
        model, dataloaders['test'], dummy_text_emb, class_words, label_to_word,
        None, config['temperature'], device, mode='eval'
    )

    # Step 6: Collect embeddings and compute metrics
    print("\n📈 Collecting test embeddings...")
    visual_emb, labels = collect_embeddings(model, dataloaders['test'], device)

    print("\n📊 Computing alignment metrics...")
    class_stats, i2t_recalls, t2i_recalls, sim_matrix = compute_alignment_metrics(
        visual_emb, labels, dummy_text_emb.cpu().numpy(), class_words, label_to_word
    )

    # Step 7: Print analysis results
    print_analysis_results(class_stats, i2t_recalls, t2i_recalls, len(labels), n_classes)
    print_example_retrievals(sim_matrix, labels, class_words, label_to_word)

    # Step 8: Create visualizations
    create_visualizations(sim_matrix, labels, class_words, label_to_word, all_test_indices)

    # Step 9: Optional OOD testing (comment out if no internet in Colab)
    try:
        print("\n🌍 Testing on OOD images (optional)...")
        ood_image_urls = {
            "Red Car": "https://images.unsplash.com/photo-1549399542-7e3f8b79c341?w=400&h=300&fit=crop",
            "Golden Retriever": "https://images.unsplash.com/photo-1552053831-71594a27632d?w=400&h=300&fit=crop",
            "Coffee Mug": "https://images.unsplash.com/photo-1517256064527-09c73fc73e38?w=400&h=300&fit=crop"
        }

        ood_images, ood_names, ood_predictions = process_ood_images(
            model, ood_image_urls, dummy_text_emb.cpu().numpy(), class_words, device
        )

        if ood_images:
            create_visualizations(
                sim_matrix, labels, class_words, label_to_word,
                all_test_indices, ood_images, ood_names, ood_predictions
            )
            has_ood = True
        else:
            has_ood = False
    except Exception as e:
        print(f"  Skipping OOD test (internet/connection issue): {e}")
        has_ood = False

    # Step 10: Print final report
    print_final_report(
        config, test_loss, test_sim, i2t_recalls, t2i_recalls, class_stats,
        len(train_idx), len(val_idx), len(all_test_indices), n_classes,
        config['batch_sizes']['train'], has_ood, history, best_epoch,
        best_val_sim, best_val_loss
    )

    print("\n🎉 Training completed successfully!")
    print(f"Model saved to: {config['save_path']}")

    return model, history


In [ ]:
# =============================================================================
# QUICK TEST FUNCTION
# =============================================================================

def quick_test():
    """Quick test to verify the model works without full training."""
    print("Running quick test...")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Create a small test
    test_dataset = CIFAR100Filtered(split="val")
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

    # Initialize model
    model = ImageEncoder(proj_dim=64, device=device)

    # Test forward pass
    images, labels = next(iter(test_loader))
    images = images.to(device)

    with torch.no_grad():
        features, projections = model(images)

    print(f"✓ Model initialized successfully")
    print(f"✓ Input shape: {images.shape}")
    print(f"✓ Features shape: {features.shape}")
    print(f"✓ Projections shape: {projections.shape}")

    # Test loss computation
    dummy_text_emb = torch.randn(4, 64).to(device)
    loss = compute_contrastive_loss(projections, dummy_text_emb, temperature=0.07)
    print(f"✓ Loss computation works: {loss.item():.4f}")

    return True


In [ ]:
# =============================================================================
# COLAB-SPECIFIC SETUP
# =============================================================================

def setup_colab():
    """Setup for Google Colab environment."""
    # Mount Google Drive (optional)
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully")

        # Change to drive directory if needed
        # os.chdir('/content/drive/MyDrive/your_folder')
    except:
        print("Not running in Colab or drive mount failed")

    # Check GPU
    if torch.cuda.is_available():
        print(f"\n✅ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("\n⚠️  No GPU available, using CPU (training will be slow)")

    # Install additional packages if needed
    try:
        import seaborn
        print("✓ All required packages are installed")
    except:
        print("Installing seaborn...")
        !pip install seaborn

In [ ]:
# =============================================================================
# RUN THE NOTEBOOK
# =============================================================================

if __name__ == "__main__":
    print("=" * 80)
    print("CIFAR-100 Image Projection Model Training")
    print("=" * 80)

    # Setup for Colab
    setup_colab()

    # Quick test first
    print("\n🧪 Running quick test...")
    if quick_test():
        print("\n✅ Quick test passed! Ready for training.")

        # Ask user if they want to train
        response = input("\nDo you want to start training? (yes/no): ").lower().strip()
        if response in ['yes', 'y', '']:
            print("\n" + "="*80)
            print("STARTING TRAINING")
            print("="*80)

            # Start training
            model, history = main()

            print("\n" + "="*80)
            print("TRAINING COMPLETE!")
            print("="*80)

            # Save the final model for neuro-symbolic planning
            print("\n💾 Saving final model for neuro-symbolic planning...")
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': {
                    'proj_dim': 64,
                    'backbone': 'mobilenet_v3_small'
                },
                'class_words': [
                    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
                    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
                    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
                    'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
                    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
                    'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
                    'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
                    'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
                    'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
                    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose',
                    'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake',
                    'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
                    'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout',
                    'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman',
                    'worm'
                ]
            }, 'best_cifar100_projection.pth')

            print("✅ Model saved as 'best_cifar100_projection.pth'")
            print("\n🎯 This model is now ready for the neuro-symbolic planning system!")

        else:
            print("\nTraining skipped. Model is ready but not trained.")
    else:
        print("\n❌ Quick test failed. Please check the implementation.")


CIFAR-100 Image Projection Model Training
Mounted at /content/drive
Google Drive mounted successfully

⚠️  No GPU available, using CPU (training will be slow)
✓ All required packages are installed

🧪 Running quick test...
Running quick test...


100%|██████████| 169M/169M [00:11<00:00, 15.3MB/s]


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 84.6MB/s]


✓ Model initialized successfully
✓ Input shape: torch.Size([4, 3, 224, 224])
✓ Features shape: torch.Size([4, 576])
✓ Projections shape: torch.Size([4, 64])
✓ Loss computation works: 3.0839

✅ Quick test passed! Ready for training.

Do you want to start training? (yes/no): Y

STARTING TRAINING
Using device: cpu

🔧 Creating dummy text embeddings for CIFAR-100 classes...

📊 Preparing CIFAR-100 datasets...
  Training samples: 40,000
  Validation samples: 10,000
  Test samples: 10,000
  Number of classes: 100

🧠 Initializing image encoder model...
  Total parameters: 1,256,288
  Trainable parameters: 329,280 (projection head only)

🚀 Starting training...

Training (max 50 epochs, patience=10)

Epoch 1/50


Eval Epoch: 100%|██████████| 79/79 [02:32<00:00,  1.93s/it, loss=1.89, sim=0.241]


  Train Loss: 2.0004, Val Loss: 2.4855, Val Sim: 0.3184
  Learning Rate: 0.000999
  ✓ New best model saved to best_cifar100_projection.pth

Epoch 2/50


Eval Epoch: 100%|██████████| 79/79 [02:29<00:00,  1.90s/it, loss=1.52, sim=0.314]


  Train Loss: 1.5167, Val Loss: 2.1861, Val Sim: 0.3607
  Learning Rate: 0.000996
  ✓ New best model saved to best_cifar100_projection.pth

Epoch 3/50


Train Epoch:   8%|▊         | 52/625 [01:00<10:39,  1.12s/it, loss=1.07, sim=0.414]